
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>



# 랩 - 회수 에이전트 구축 및 등록

## 개요

이 랩에서는 Databricks Mosaic AI를 사용하여 프로덕션 준비 완료된 검색 에이전트를 구축하고 등록하게 됩니다. LangChain 기반 에이전트를 만들어 대형 언어 모델과 AI Search 인덱스를 결합하고, 관찰 가능성을 위한 MLflow 추적을 구현하며, 적절한 버전 관리와 별칭으로 에이전트를 Unity Catalog's Model Registry에 등록하게 됩니다.

## 학습 목표

이 실험실이 끝날 때쯤이면 다음과 같은 일을 할 수 있게 될 것입니다:

1. LangChain에 대해 MLflow 추적을 활성화하여 에이전트 동작을 모니터링하세요.
1. LangChain과 AI Search 통합을 사용하여 검색 에이전트를 구축하세요.
1. 에이전트 실행 추적을 분석하여 성능 특성을 파악하세요.
1. "agent as code" 패턴에 따라 Python 파일에 에이전트 코드를 작성합니다.
1. 에이전트 모델을 Unity Catalog로 등록하고 별칭을 사용하세요.
1. 등록된 모델을 테스트하여 기능을 검증합니다.

## 요구 사항

- 미리 생성된 **AI Search 엔드포인트**. 이것은 미리 생성되었습니다.
- **서버리스 Compute (환경 버전 5)**. [여기](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version)에 따라 적절한 환경 버전을 선택하세요.


**📌 당신의 태스크: 이 실험실에서 당신의 태스크는 섹션을 적절한 코드로 교체 `<FILL_IN>` 하는 것입니다.**

## 준비

아래 코드를 실행하여 필요한 라이브러리를 설치하고 교실 환경을 구성하세요. 이 단계는 모든 의존성이 사용 가능하고 워크스페이스가 데모 준비가 완료되도록 보장합니다.

In [0]:
%run ../Includes/Classroom-Setup-04 $section="lab"

## A. LangChain와 함께 검색 에이전트 구축

이 섹션에서는 **LangChain**를 사용하여 검색 에이전트를 만듭니다. 에이전트는 Unity catalog의 AI Search를 도구로 활용하여 사용자 질문에 답변할 때 동적으로 관련 컨텍스트를 검색할 수 있습니다.

**"agent as code"** 방식을 따르면, 에이전트 구현을 Python 파일(`agent.py`)로 작성해야 합니다. 이 방법은 MLflow 모델을 기록할 때 권장됩니다.

**Dataset 정보:** AI Search 인덱스에는 Orion이라는 이름의 로봇을 위한 가상의 로봇 제조사의 데이터가 포함되어 있습니다. 문서에는 내부 설계 매뉴얼에서 출처를 둔 맥락 기반 답변, 규정 준수 문서, 유지보수 가이드가 포함됩니다.

### A1. 태스크 1 - MLflow 추적 활성화

에이전트를 만들기 전에 **LangChain에 대해 MLflow 추적을 활성**화하여 에이전트의 입력, 도구 사용, 출력을 자세히 관찰해야 합니다.

**당신의 태스크:**

1. 적절한 방법으로 MLflow 자동 로그를 활성화하세요 LangChain

In [0]:
## MLflow를 가져오고 LangChain의 자동 로그를 활성화하세요

<FILL_IN>

In [0]:
%skip
import mlflow
mlflow.langchain.autolog()

### A2. 태스크 2 - LangChain 에이전트 생성

이제 AI Search 리트리버 도구를 사용해 Orion 지식 베이스에 접근하는 LangChain 에이전트를 만들 것입니다.

**귀하의 작업:**

1. LLM 엔드포인트 이름을 `"databricks-claude-sonnet-4-6"`로 정의하십시오.
1. `build_agent` 함수를 다음과 같이 완성하십시오:
   * 제공된 엔드포인트와 `ChatDatabricks`를 사용하여 `max_tokens=300` 모델 생성.
   * 다음을 사용하여 `VectorSearchRetrieverTool` 생성:
     - `name="orion_knowledge_search_lab"`
     - parameter에서 `index_name`
     - `description="Search Orion knowledge base for relevant information"`
     - parameter에서 `num_results` (5개 결과)
   * 제공된 시스템 프롬프트를 사용해.
   * 모델, 도구 목록, 시스템 프롬프트, 체크포인터를 사용하여 `create_agent` 에이전트를 생성하기.
1. 에이전트에게 "오리온이란 무엇인가?"라는 질문으로 테스트해 보세요.

In [0]:
from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

llm_endpoint_name = <FILL_IN>

def build_agent(llm_endpoint:str, index_name: str, num_results: int = 3):
    model = <FILL_IN>

    vs_tool = <FILL_IN>

    ## 선택 사항: 메모리에 저장하는 세이버를 사용해 에이전트 상태를 저장하세요
    checkpointer = <FILL_IN>

    system_prompt = """당신은 오리온 지식 조수(OKA)입니다. 엔지니어와 기술진에게 적합한 명확하고 전문적이며 사실적인 어조로 답변하세요. 오리온 내부 문서에서 검증된 정보만 사용하고, 가능한 경우 출처 참고 자료를 포함하세요. 답을 찾지 못하면 명확히 말하고 관련 섹션이나 다음 단계를 제안하세요. 추측하거나 가정하거나 제공된 맥락 밖의 정보를 제공하지 마십시오."""

    agent = <FILL_IN>
    return agent

## 빠른 스모크 테스트
agent = build_agent(llm_endpoint_name, index_name, 3)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Orion?"}]}
)
print(response['messages'][-1].content)

In [0]:
%skip

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

llm_endpoint_name = "databricks-claude-sonnet-4-6"

def build_agent(llm_endpoint:str, index_name: str, num_results: int = 5):
    model = ChatDatabricks(
        endpoint=llm_endpoint,
        max_tokens=300,
    )

    vs_tool = VectorSearchRetrieverTool(
        name="orion_knowledge_search_lab",
        index_name=index_name,
        description="Search Orion knowledge base for relevant information",
        num_results=num_results,
    )

    system_prompt = """You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone appropriate for engineers and technical staff. Use only verified information from Orion's internal documents, and include source references when available. If the answer cannot be found, clearly state that and suggest related sections or next steps. Do not speculate, make assumptions, or provide information outside the provided context."""

    agent = create_agent(
        model=model, 
        tools=[vs_tool], 
        system_prompt=system_prompt,
    )
    return agent

# Quick smoke test
agent = build_agent(llm_endpoint_name, vs_index_name, 3)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Orion?"}]}
)
print(response['messages'][-1].content)

### A3. 태스크 3 - MLflow 추적 UI 검토

MLflow 추적 UI는 에이전트의 실행 및 도구 사용에 대한 포괄적인 뷰를 제공합니다. 위 출력물은 트레이싱 UI를 보여줍니다.

**귀하의 태스크:**

1. 실행 타임라인에서 가장 긴 단계의 이름을 찾아보세요.
1. 그 단계를 담당한 도구나 모델을 식별하세요.
1. 에이전트 실행에 사용된 토큰의 총 수(입력, 출력, 총 토큰)를 확인하세요.


## B. 에이전트 로깅 및 Model Registry에 등록

이 섹션에서는 에이전트를 Python 파일에 작성하고 Unity catalog의 Model Registry에 등록하여 프로덕션 환경에 준비합니다. 새 구성 파일을 생성하는 대신 데모 노트북의 구성 파일을 사용할 것입니다.

### B1. 에이전트 코드 및 구성 파일 작성

다음 단계들은 에이전트를 기록하고 등록하는 데 필요한 파일을 생성합니다:

- **`agent-lab.py`**: 이 파일은 MLflow `pyfunc`로 에이전트 로직을 감싸 요청 지원을 `ResponseAgent` 가능하게 합니다.
- **`agent-config-lab.yaml`**: 이 파일에는 에이전트 구성이 포함되어 있습니다.

이 섹션에서는 별도의 조치가 필요하지 않습니다. 필요한 파일을 생성하기 위해 코드를 실행하기만 하면 됩니다.


In [0]:
%%writefile agent-lab.py
import os
from typing import Any, Dict, List

import yaml
import mlflow
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import ResponsesAgentRequest, ResponsesAgentResponse

from uuid import uuid4

from langchain.agents import create_agent
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool

# YAML 파일에서 에이전트 구성 불러오기
def _load_config(path: str = "agent-config.yaml") -> Dict[str, Any]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Config file not found at '{path}'")
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    llm_endpoint = cfg.get("llm_endpoint_name")
    vs = cfg.get("vector_search", {}) or {}
    index_name = vs.get("index_name")
    num_results = int(vs.get("num_results", 5))
    if not llm_endpoint or not index_name:
        raise ValueError("Missing 'llm_endpoint_name' or 'vector_search.index_name' in agent-config-lab.yaml")
    return {
        "llm_endpoint_name": llm_endpoint,
        "vs_index_name": index_name,
        "vs_num_results": num_results,
    }

# LLM과 AI Search 도구로 LangChain 에이전트를 구축합니다
def build_agent(llm_endpoint: str, index_name: str, num_results: int = 5):
    model = ChatDatabricks(endpoint=llm_endpoint, max_tokens=300)
    vs_tool = VectorSearchRetrieverTool(
        name="orion_knowledge_search",
        index_name=index_name,
        description="Search Orion knowledge base for relevant information",
        num_results=num_results,
    )

    system_prompt = (
        "You are the Orion Knowledge Assistant (OKA). Respond in a clear, professional, and factual tone "
        "appropriate for engineers and technical staff. Use only verified information from Orion's internal "
        "documents, and include source references when available. If the answer cannot be found, clearly state "
        "that and suggest related sections or next steps. Do not speculate, make assumptions, or provide "
        "information outside the provided context."
    )
    agent = create_agent(
        model=model,
        tools=[vs_tool],
        system_prompt=system_prompt
    )
    return agent

# 대화에서 마지막 사용자 메시지 추출하기
def _last_user_text(messages: List[Dict[str, Any]]) -> str:
    user_msgs = [m for m in messages if (m.get("role") == "user")]
    return str(user_msgs[-1].get("content", "")) if user_msgs else str(messages[-1].get("content", ""))

# MLflow ResponsesAgent implementation for LangChain agent
class LangChainResponsesAgent(ResponsesAgent):
    def __init__(self):
        cfg = _load_config()
        self._cfg = cfg
        self._agent = build_agent(
            llm_endpoint=cfg["llm_endpoint_name"],
            index_name=cfg["vs_index_name"],
            num_results=cfg["vs_num_results"],
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        msgs = [m.model_dump() for m in request.input]  # [{'role': 'user'|'assistant', 'content': '...'}, ...]
        _ = _last_user_text(msgs) if msgs else ""

        result = self._agent.invoke(
            {"messages": msgs}
        )
        # Extract agent response text
        try:
            text = result["messages"][-1].content
        except Exception:
            text = str(result)

        return ResponsesAgentResponse(
            output=[self.create_text_output_item(text, str(uuid4()))],
            custom_outputs=request.custom_inputs,
        )

# 모델을 설정하여 MLflow. 이는 agent-as-code 방식을 사용할 때 필요합니다
AGENT = LangChainResponsesAgent()
mlflow.models.set_model(AGENT)

In [0]:
import yaml

def create_config(llm_endpoint_name: str, index_name: str, num_results: int = 3):
    """Create a minimal YAML config for the agent."""
    config = {
        "llm_endpoint_name": llm_endpoint_name,
        "vector_search": {
            "index_name": index_name,
            "num_results": num_results
        }
    }
    return config


# 설정 파일 생성
llm_endpoint_name = "databricks-claude-sonnet-4-6"

agent_config = create_config(llm_endpoint_name, vs_index_name)

# YAML 파일을 작성하세요 (agent.py이 나중에 읽을 수 있도록)
with open("agent-config-lab.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(agent_config, f, sort_keys=False)

print("✅ Config file written: agent-conf.yaml")
print(yaml.safe_dump(agent_config, sort_keys=False))


### B2. 태스크 4 - 에이전트 모델을 Unity Catalog에 별칭과 함께 등록합니다

이제 에이전트 모델을 Unity Catalog's Model Registry에 등록하게 됩니다. 이 기능은 로그와 등록을 하나의 워크플로로 통합합니다. 등록된 모델 버전에도 **별칭**을 추가해야 합니다.

**귀하의 태스크:**

1. 모델 리소스(AI Search 인덱스 및 서빙 엔드포인트)를 정의하세요.
1. `mlflow.pyfunc.log_model()` 을 사용하여 에이전트 모델을 다음과 같이 로깅하십시오:
   * 모델명: **`"orion_knowledge_assistant_lab"`**
   * Python 모델: **`"agent-lab.py"`**
   * 코드 경로: **`["agent-config-lab.yaml"]`**
   * **모델 이름**을 등록하여 UC 모델 레지스트리에 변환.
   * 입력 예시
   * 필수 pip 패키지
   * 자료
1. 등록된 모델 버전에 별칭을 설정하세요.

**힌트:** 별칭 설정 방법을 배우려면 [MLflow Model Registry 문서](https://docs.databricks.com/aws/en/machine-learning/manage-model-lifecycle/#use-model-aliases)를 참고하세요. 

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from importlib.metadata import version as get_version
import mlflow

## 1단계: 리소스를 정의하세요
resources = <FILL_IN>

print("Resources defined:")
for resource in resources:
    print(f"  - {resource}")

## 2단계: 모델 구성 정의하세요
model_name = "orion_knowledge_assistant"
tags_to_register = {
    "model_type": "retrieval_agent",
    "framework": "langchain",
    "use_case": "orion_knowledge_base"
}

input_example = {
    "input": [
        {"role": "user", "content": "What is Orion?"}
    ]
}

## 3단계: 모델을 로그하십시오
with mlflow.start_run():
    mlflow.set_tags(tags_to_register)
    
    logged_agent_info = <FILL_IN>
    
    model_uri = logged_agent_info.model_uri
    
print(f"✅ Model logged successfully!")
print(f"Model URI: {model_uri}")

## 4단계: 모델을 Unity Catalog에 등록하세요
mlflow.set_registry_uri("databricks-uc")
UC_MODEL_NAME = f"{catalog}.{schema}.orion_knowledge_assistant_lab"

uc_registered_model_info = <FILL_IN>

print(f"✅ Model registered successfully to Unity Catalog!")
print(f"Model Name: {UC_MODEL_NAME}")
print(f"Version: {uc_registered_model_info.version}")

## 5단계: 등록된 모델 버전에 별칭을 설정하세요
## 별칭 설정 방법을 배우려면 문서를 참고하세요
client = <FILL_IN>
<FILL_IN>

print(f"✅ Alias 'Champion' set for version {uc_registered_model_info.version}")

In [0]:
%skip
from mlflow.models.resources import DatabricksVectorSearchIndex, DatabricksServingEndpoint
from importlib.metadata import version as get_version
import mlflow

# 1단계: 리소스를 정의하세요
resources = [
    DatabricksVectorSearchIndex(index_name=vs_index_name),
    DatabricksServingEndpoint(endpoint_name=llm_endpoint_name)
]

print("Resources defined:")
for resource in resources:
    print(f"  - {resource}")

# Step 2: Define model configuration
model_name = "orion_knowledge_assistant_lab"
UC_MODEL_NAME = f"{catalog}.{schema}.orion_knowledge_assistant_lab"

input_example = {
    "input": [
        {"role": "user", "content": "What is Orion?"}
    ]
}

# 3단계: 모델을 한 단계로 로그 및 등록하세요
mlflow.set_registry_uri("databricks-uc")
with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name=model_name,
        python_model="agent-lab.py",
        code_paths=["agent-config-lab.yaml"],
        input_example=input_example,
        pip_requirements=[
            f"databricks-vectorsearch=={get_version('databricks-vectorsearch')}",
            f"databricks-langchain=={get_version('databricks-langchain')}",
            f"langchain=={get_version('langchain')}",
            f"mlflow=={get_version('mlflow')}",
        ],
        resources=resources,
        registered_model_name=UC_MODEL_NAME
    )
    model_uri = logged_agent_info.model_uri
    model_version = logged_agent_info.registered_model_version

print(f"✅ Model logged and registered successfully!")
print(f"Model URI: {model_uri}")
print(f"Model Name: {UC_MODEL_NAME}")
print(f"Version: {model_version}")

# 4단계: 등록된 모델 버전에 별칭을 설정하세요
client = mlflow.MlflowClient()
client.set_registered_model_alias(
    name=UC_MODEL_NAME,
    alias="Champion",
    version=model_version
)

print(f"✅ Alias 'Champion' set for version {model_version}")

## C. 등록된 모델 테스트

이 마지막 섹션에서는 그것이 올바르게 작동하는지 검증하기 위해 등록된 모델을 테스트할 것입니다. Unity Catalog에서 모델을 불러와서 예측을 해야 합니다.

### C1. 태스크 5 - 등록된 모델 테스트

이제 에이전트가 Unity Catalog에 등록되었으니, 기능을 확인하기 위해 그것을 테스트할 것입니다.

**귀하의 임무:**

1. 이전 단계의 모델 URI를 사용해 예측을 하세요.
1. 모델과 함께 기록된 입력 예제를 사용하세요.
1. 에이전트의 답변을 출력하세요.


In [0]:
## 예측을 통해 등록된 모델을 검증합니다

query = <FILL_IN>

result = <FILL_IN>

print("Agent Response:")
print(result)

In [0]:
%skip
import mlflow

query = {
    "input": [
        {"role": "user", "content": "What are the safety procedures for the Orion?"}
    ]
}

## Make a prediction using the model URI
result = mlflow.models.predict(
    model_uri=model_uri,
    input_data=query,
    env_manager="uv",
)

print("Agent Response:")
print(result)

### C2. 태스크 6 - Model Registry UI 탐색

에이전트가 Unity Catalog에 성공적으로 등록되었으니, 이제 Model Registry UI를 탐색해 모델을 관리, 모니터링, 관리하는 방법을 이해할 수 있습니다.

**사용자의 임무:**

- Model Registry UI의 네 가지 주요 탭을 식별하고 그 목적을 설명하세요.
- 모델과 함께 아티팩트로 기록된 **모델 요구사항 파일**을 찾아보세요.
- 등록된 모델 버전에 설정한 **별칭**을 찾아보세요.
- 에이전트 호출에 대한 **실행 추적**을 검토합니다.
- model registry UI가 어떻게 **모델 수명주기 관리**와 **거버넌스**를 지원하는지 요약하세요.

## D. 요약

Databricks Mosaic AI를 사용하여 프로덕션 준비 가능한 검색 에이전트를 성공적으로 구축, 로그, 등록하셨습니다.

이 랩에서 여러분은:

* **LangChain에 대해 MLflow 추적**을 활성화하여 에이전트의 동작과 실행을 모니터링했습니다.
* LangChain을 사용하여 AI Search 통합으로 **검색 에이전트를 만들**었습니다.
* **에이전트 트레이스를 분석**하여 성능 특성을 확인하고 실행 흐름을 이해했습니다.
* **에이전트 코드**를 Python 파일에 '코드로서의 에이전트' 패턴으로 작성했습니다.
* **에이전트**를 Unity Catalog's Model Registry에 **등록**했고, 적절한 버전과 별칭을 사용했습니다.
* **등록된 모델을 테스트**하여 기능을 검증합니다.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>